# Лабораторная № 5, вариант 5 — отчёт о качестве данных

**Источник:** учебный магазин (WooCommerce), по умолчанию `scrapingcourse.com/ecommerce`.
**Что анализируем:** базу `data/crawl.db` (звено 400 пайплайна) и файлы статистики прогонов `data/stats_*.json` (метод `closed()` паука).

Блокнот выполняется после реального прогона (`./run_real.sh`) целиком: *Kernel → Restart & Run All*. Все цифры в нём считаются из файлов, вручную ничего не вписывается.

| Раздел | Что считаем |
|---|---|
| 1 | сводка итогового прогона: запросы, коды ответа, повторы, трафик, длительность |
| 2 | число записей и уникальных ключей (SKU) |
| 3 | полнота каждого поля, % |
| 4 | отброшенные записи и причины (из статистики) |
| 5 | коды ответа и повторы по всем прогонам |
| 6 | аналитические агрегаты: цены по категориям, скидки, распределение цен |
| 7 | демонстрация JOBDIR и HTTPCACHE |
| 8 | smoke-check деградации |
| 9 | сверка с результатом n8n |

In [ ]:
import json, sqlite3, subprocess, sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)

ROOT = Path.cwd().parent if Path.cwd().name == "quality" else Path.cwd()
DATA = ROOT / "data"
DB = DATA / "crawl.db"
assert DB.exists(), f"Нет базы {DB}: сначала выполните прогон (./run_real.sh)"

def load_stats(path):
    s = json.loads(Path(path).read_text(encoding="utf-8"))
    s["_file"] = Path(path).name
    return s

stats_all = sorted((load_stats(p) for p in DATA.glob("stats_*.json")), key=lambda s: s.get("run_no", 0))
main_runs = [s for s in stats_all if str(s.get("run_label", "")).startswith("main")]
MAIN = main_runs[-1] if main_runs else stats_all[-1]
print("Файлов статистики:", len(stats_all))
print("Итоговый прогон  :", MAIN["_file"], "| run_id =", MAIN.get("run_id"), "| источник:", MAIN.get("start_url"))

## 1. Сводка итогового прогона

In [ ]:
codes = {k.rsplit("/", 1)[-1]: v for k, v in MAIN.items() if k.startswith("downloader/response_status_count/")}
summary = pd.DataFrame([
    ("Причина остановки", MAIN.get("close_reason")),
    ("Предохранитель CLOSESPIDER_ITEMCOUNT", MAIN.get("settings/CLOSESPIDER_ITEMCOUNT")),
    ("Запросов (downloader/request_count)", MAIN.get("downloader/request_count")),
    ("из них robots.txt", MAIN.get("robotstxt/request_count")),
    ("Страниц каталога (pages/list)", MAIN.get("pages/list")),
    ("Коды ответа", ", ".join(f"{c}: {n}" for c, n in sorted(codes.items()))),
    ("Повторов (retry/count)", MAIN.get("retry/count", 0)),
    ("Ошибок в errback", MAIN.get("errback/total", 0)),
    ("Трафик ответов, МБ (на проводе)", round(MAIN.get("downloader/response_bytes", 0) / 1_048_576, 3)),
    ("Трафик после распаковки, МБ", round(MAIN.get("httpcompression/response_bytes", 0) / 1_048_576, 3)),
    ("Длительность, с", round(MAIN.get("elapsed_time_seconds", 0), 1)),
    ("Записей прошло пайплайн (item_scraped_count)", MAIN.get("item_scraped_count")),
    ("Записей отброшено (item_dropped_count)", MAIN.get("item_dropped_count", 0)),
    ("Записано в БД (db/rows_written)", MAIN.get("db/rows_written")),
    ("Записей в минуту", round(MAIN.get("item_scraped_count", 0) / max(MAIN.get("elapsed_time_seconds", 1), 1e-9) * 60, 1)),
], columns=["метрика", "значение"])
summary

## 2. Записи и уникальные ключи

In [ ]:
conn = sqlite3.connect(DB)
df = pd.read_sql_query("SELECT * FROM products", conn)
cats = pd.read_sql_query("SELECT * FROM product_categories", conn)
df["price_num"] = df["price_cents"] / 100
df["categories_list"] = df["categories"].apply(json.loads)
print("Записей в таблице products  :", len(df))
print("Уникальных SKU (бизнес-ключ):", df["sku"].nunique())
print("Уникальных URL карточек     :", df["product_url"].nunique())
print("Страниц каталога с товарами :", df["list_page_no"].nunique())
print("Источник артикула (sku_source):", df["sku_source"].value_counts().to_dict())
df[["sku", "name", "price", "price_max", "regular_price", "on_sale", "currency", "categories", "image_url"]].head(5)

## 3. Полнота полей, %

Пустым считается `NULL`, пустая строка и пустой JSON-массив. Обязательные поля варианта 5: название, цена, артикул, категории, описание, изображение.

In [ ]:
def filled(series):
    return series.notna() & ~series.astype(str).isin(["", "[]", "None"])

variant_fields = ["name", "price", "sku", "categories", "description", "image_url"]
extra_fields = ["short_description", "tags", "regular_price", "price_max", "product_id"]
completeness = pd.DataFrame({
    "поле": variant_fields + extra_fields,
    "обязательное (вариант 5)": ["да"] * len(variant_fields) + ["нет"] * len(extra_fields),
    "заполнено, %": [round(filled(df[c]).mean() * 100, 1) for c in variant_fields + extra_fields],
})
completeness

## 4. Отброшенные записи и причины

Счётчики `dropped/*` ведут звенья пайплайна (100 — нет обязательных полей, 200 — неразборчивая цена, 300 — дубль SKU). Пустые значения считает звено 100 до отбраковки (`quality/empty/*`).

In [ ]:
drops = {k: v for k, v in MAIN.items() if k.startswith("dropped/")}
empties = {k.replace("quality/empty/", ""): v for k, v in MAIN.items() if k.startswith("quality/empty/")}
seen = MAIN.get("quality/items_seen", 0)
print("Карточек дошло до пайплайна:", seen)
display(pd.DataFrame(sorted(drops.items()), columns=["причина", "записей"]))
display(pd.DataFrame([{"поле": f, "пустых до очистки": n, "доля, %": round(n / seen * 100, 2) if seen else None}
                      for f, n in sorted(empties.items())]))
fallbacks = {k: v for k, v in MAIN.items() if k.startswith(("fields/", "clean/"))}
print("Срабатывания фолбэков и правок очистки:", fallbacks or "нет")

## 5. Коды ответа, повторы, трафик — по всем прогонам

In [ ]:
rows = []
for s in stats_all:
    row = {"файл": s["_file"], "метка": s.get("run_label"), "причина": s.get("close_reason"),
           "запросов": s.get("downloader/request_count", 0), "записей": s.get("item_scraped_count", 0),
           "повторов": s.get("retry/count", 0), "КБ": round(s.get("downloader/response_bytes", 0) / 1024, 1),
           "сек": round(s.get("elapsed_time_seconds", 0), 1)}
    for k, v in s.items():
        if k.startswith("downloader/response_status_count/"):
            row["HTTP " + k.rsplit("/", 1)[-1]] = v
    rows.append(row)
runs_df = pd.DataFrame(rows).fillna(0)
runs_df

In [ ]:
code_cols = [c for c in runs_df.columns if c.startswith("HTTP ")]
ax = runs_df.set_index("файл")[code_cols].plot(kind="bar", stacked=True, figsize=(10, 4))
ax.set_ylabel("ответов"); ax.set_title("Распределение кодов ответа по прогонам")
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

## 6. Аналитические агрегаты

In [ ]:
by_cat = (cats.merge(df[["sku", "price_num", "on_sale"]], on="sku")
              .groupby("category")
              .agg(товаров=("sku", "count"), средняя_цена=("price_num", "mean"),
                   медиана=("price_num", "median"), мин=("price_num", "min"), макс=("price_num", "max"),
                   доля_со_скидкой=("on_sale", "mean"))
              .sort_values("товаров", ascending=False))
by_cat["средняя_цена"] = by_cat["средняя_цена"].round(2)
by_cat["доля_со_скидкой"] = (by_cat["доля_со_скидкой"] * 100).round(1)
print("Агрегат 1. Цены по категориям (товар может входить в несколько категорий):")
by_cat.head(15)

In [ ]:
print("Агрегат 2. Скидки и вариативные товары")
print(f"  со скидкой (on_sale)      : {int(df['on_sale'].sum())} из {len(df)} ({df['on_sale'].mean():.1%})")
disc = df[df["on_sale"] == 1].copy()
if len(disc):
    disc["скидка, %"] = (1 - disc["price_num"] / disc["regular_price"].astype(float)) * 100
    print(f"  средняя глубина скидки    : {disc['скидка, %'].mean():.1f} %")
print(f"  с диапазоном цен (price_max): {df['price_max'].notna().sum()}")
print()
print("Агрегат 3. Распределение цен (квантили), валюта:", df["currency"].unique().tolist())
print(df["price_num"].describe(percentiles=[.1, .25, .5, .75, .9]).round(2).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["price_num"].plot(kind="hist", bins=20, ax=axes[0], edgecolor="white")
axes[0].set_title("Распределение цен"); axes[0].set_xlabel("цена")
by_cat.head(10)["средняя_цена"].sort_values().plot(kind="barh", ax=axes[1])
axes[1].set_title("Средняя цена по категориям (топ-10 по числу товаров)")
plt.tight_layout(); plt.show()

## 7. Демонстрация JOBDIR и HTTPCACHE

Сводку пишет `tools/demo_resume_cache.py` (вызывается из `run_real.sh`).

In [ ]:
demo = DATA / "demo" / "demo_summary.json"
if demo.exists():
    d = json.loads(demo.read_text(encoding="utf-8"))
    if d.get("fast_mode_local_mock"):
        print("ВНИМАНИЕ: сводка получена на локальной имитации, а не на реальном сайте")
    j = d["jobdir"]
    display(pd.DataFrame([{"прогон": "1 (прерван CLOSESPIDER_ITEMCOUNT)", **j["run1"]},
                          {"прогон": "2 (возобновление)", **j["run2"]}]))
    print("Отпечатков в requests.seen после прогона 1:", j["fingerprints_in_requests_seen"])
    print("Уникальных SKU за два прогона:", j["unique_sku_total"], "| повторно собрано во втором:", j["sku_repeated_in_run2"])
    display(pd.DataFrame(j["jobstate_files"]))
    display(pd.DataFrame(d["httpcache"]))
else:
    print("Сводки демонстрации нет: выполните python3 tools/demo_resume_cache.py")

## 8. Smoke-check деградации между прогонами

In [ ]:
for extra in ([], ["--self-test"]):
    r = subprocess.run([sys.executable, str(ROOT / "quality" / "smoke_check.py"), *extra],
                       capture_output=True, text=True, timeout=120)
    print(r.stdout.strip(), f"\n(код возврата {r.returncode})\n")

## 9. Сверка с результатом n8n

In [ ]:
csv = ROOT / "nocode" / "nocode_result.csv"
if csv.exists():
    n8n = pd.read_csv(csv, encoding="utf-8-sig")       # n8n пишет CSV с BOM
    print("Записей в nocode_result.csv:", len(n8n), "| столбцы:", list(n8n.columns))
    if "sku" in n8n.columns:
        both = set(n8n["sku"].astype(str)) & set(df["sku"])
        print(f"SKU, найденные и в n8n, и в Scrapy: {len(both)} из {n8n['sku'].nunique()}")
        if "price" in n8n.columns:
            m = n8n.astype({"sku": str}).merge(df[["sku", "price_num"]], on="sku")
            diff = (m["price"].astype(float) - m["price_num"]).abs() > 0.005
            print("Расхождений цены между n8n и Scrapy:", int(diff.sum()))
    display(n8n.head())
else:
    print("nocode/nocode_result.csv ещё нет: выполните workflow в n8n (nocode/INSTRUCTIONS.md)")
conn.close()